# 08 - Gold document lineage (Phase 8)

Promotes **Delivery**, **Invoice** and **Receivable** to first-class Gold document tables, and
builds three explicit **edge tables** so Fabric Graph can traverse

`OrderLine -> Delivery -> Invoice -> Receivable`

### Why edge tables and not the ontology relationships

Ontology relationships do **not** become graph edges. Semantic-model relationships flow into the
ontology automatically, but a graph edge is a separate construct materialised from a **source
table** with an origin key column and a target key column. Nothing warns you - the graph simply
renders disconnected nodes. Phase 6 hit exactly this.

### Why not traverse gld_fact_document_flow

It is the SAP-faithful structure, but every row is a generic (PrecedingDocId, SubsequentDocId)
pair spanning four document categories, so it needs filtering and self-joins to yield one hop.
Silver already carries the links directly - `slv_delivery_item` holds `SalesOrderId`/`SalesOrderItem`
and `slv_billing_item` holds `DeliveryId`. Purpose-built edge tables are cheaper and unambiguous.
`gld_fact_document_flow` stays as the audit-faithful record.

**Prerequisite:** attach **OntologyDataLH** as the default lakehouse and confirm the pin glyph.

In [ ]:
from pyspark.sql import functions as F, types as T
import re

SILVER, GOLD = "slv_", "gld_"

def dbl(c):
    """Gold contract: no DecimalType anywhere - Fabric Graph returns null for Decimal."""
    return F.col(c).cast("double")

def opt(df, *candidates):
    """Return the first candidate column present, else None.

    slv_delivery_item folds its header in via a left join, so the exact header column names are
    resolved at runtime rather than assumed. Guessing them is how you get a notebook that
    imports cleanly and then fails on cell 6.
    """
    for c in candidates:
        if c in df.columns:
            return c
    return None

print("Spark ready")

## 1. Inspect the Silver schemas we depend on

Run this before anything else - the delivery header column names are resolved from it.

In [ ]:
for t in ["delivery_item", "billing_item", "ar_item"]:
    cols = spark.table(f"{SILVER}{t}").columns
    print(f"{SILVER}{t}  ({len(cols)} cols)")
    print("   " + ", ".join(cols))
    print()

## 2. Node table - Delivery

Document grain: one row per `DeliveryId`, keyed for the graph.

In [ ]:
dli = spark.table(f"{SILVER}delivery_item")

c_gi   = opt(dli, "GoodsIssueDate", "ActualGoodsIssueDate", "GIDate")
c_ddat = opt(dli, "DeliveryDate", "PlannedDeliveryDate", "DeliveryCreatedDate")
c_type = opt(dli, "DeliveryType")
c_ship = opt(dli, "ShipToId", "ShipTo", "CustomerId")
print("resolved delivery header columns ->",
      {"goodsIssue": c_gi, "deliveryDate": c_ddat, "type": c_type, "shipTo": c_ship})

aggs = [F.countDistinct("DeliveryItem").alias("ItemCount"),
        F.sum(dbl("DeliveredQty")).alias("DeliveredQty"),
        F.round(F.sum(dbl("DeliveryNetValue")), 2).alias("DeliveryNetValue"),
        F.countDistinct("SalesOrderId").alias("SalesOrderCount"),
        F.first("Plant", True).alias("Plant")]
for alias, src in [("GoodsIssueDate", c_gi), ("DeliveryDate", c_ddat),
                   ("DeliveryType", c_type), ("ShipToId", c_ship)]:
    if src:
        aggs.append(F.first(src, True).alias(alias))

gld_delivery = dli.groupBy("DeliveryId").agg(*aggs)
print("delivery nodes:", gld_delivery.count())
gld_delivery.show(3, truncate=False)

## 3. Node table - Invoice

One row per `BillingDocId`. Header attributes are certain here: Silver joins `billing_hdr` explicitly.

In [ ]:
bli = spark.table(f"{SILVER}billing_item")

gld_invoice = (bli.groupBy("BillingDocId").agg(
        F.countDistinct("BillingItem").alias("ItemCount"),
        F.sum(dbl("BilledQty")).alias("BilledQty"),
        F.round(F.sum(dbl("BillingItemNetValue")), 2).alias("BilledNetValue"),
        F.round(F.sum(dbl("BillingItemCost")), 2).alias("BilledCost"),
        F.first("BillingType", True).alias("BillingType"),
        F.first("BillingDate", True).alias("BillingDate"),
        F.first("CompanyCode", True).alias("CompanyCode"),
        F.first("SalesOrg", True).alias("SalesOrg"),
        F.first("PayerId", True).alias("PayerId"),
        F.first("SoldToId", True).alias("SoldToId"),
        F.first("DocCurrency", True).alias("DocCurrency"),
        F.first("PaymentTerms", True).alias("PaymentTerms"),
        F.first("AccountingDocId", True).alias("AccountingDocId"),
        F.first("IsCancelled", True).alias("IsCancelled"),
        F.first("IsCreditMemo", True).alias("IsCreditMemo"),
        F.countDistinct("DeliveryId").alias("DeliveryCount"))
    .withColumn("BillingDateKey",
                F.date_format("BillingDate", "yyyyMMdd").cast("int")))

print("invoice nodes:", gld_invoice.count())
gld_invoice.show(3, truncate=False)

## 4. Node table - Receivable

Built from the **Gold** AR fact, not Silver, so it inherits the conformed columns and the EUR
conversion from Gold revision 3.

`gld_fact_accounts_receivable` is keyed by a four-part composite
(CompanyCode, AccountingDocId, FiscalYear, LineItem). Fabric Graph edges bind on a **single**
key column, and composite keys were never verified in Phase 6, so this node gets an explicit
single-column surrogate `ArItemKey`.

In [ ]:
ar = spark.table(f"{GOLD}fact_accounts_receivable")

gld_receivable = (ar
    .withColumn("ArItemKey", F.concat_ws("-", "CompanyCode", "AccountingDocId",
                                         "FiscalYear", "LineItem"))
    .select("ArItemKey", "CompanyCode", "CustomerId", "AccountingDocId", "FiscalYear",
            "LineItem", "BillingDocRef", "PostingDate", "DueDate", "ClearingDate",
            "Currency", "AmountLocal", "AmountEUR", "DaysToPay", "DaysOverdue",
            "IsOpen", "IsOverdue", "SourceTable"))

n, d = gld_receivable.count(), gld_receivable.select("ArItemKey").distinct().count()
print(f"receivable nodes: {n:,}   distinct keys: {d:,}")
assert n == d, "ArItemKey is not unique - it cannot be a graph node key"
print("PASS  ArItemKey is unique")

## 5. Edge tables

Each has exactly one origin key column and one target key column, which is the shape the Graph
**Add edge** dialog expects.

In [ ]:
# OrderLine -> Delivery. Grain: delivery item.
edge_od = (spark.table(f"{SILVER}delivery_item")
    .withColumn("OrderLineKey", F.concat_ws("-", "SalesOrderId", "SalesOrderItem"))
    .filter(F.col("SalesOrderId").isNotNull() & F.col("DeliveryId").isNotNull())
    .select("OrderLineKey", "DeliveryId", "DeliveryItem",
            dbl("DeliveredQty").alias("DeliveredQty"),
            F.round(dbl("DeliveryNetValue"), 2).alias("DeliveryNetValue")))

# Delivery -> Invoice. Grain: billing item that references a delivery.
edge_di = (spark.table(f"{SILVER}billing_item")
    .filter(F.col("DeliveryId").isNotNull() & (F.trim(F.col("DeliveryId")) != ""))
    .select("DeliveryId", "BillingDocId", "BillingItem",
            dbl("BilledQty").alias("BilledQty"),
            F.round(dbl("BillingItemNetValue"), 2).alias("BilledNetValue")))

# Invoice -> Receivable, via the AR billing reference.
edge_ir = (gld_receivable
    .filter(F.col("BillingDocRef").isNotNull() & (F.trim(F.col("BillingDocRef")) != ""))
    .select(F.col("BillingDocRef").alias("BillingDocId"), "ArItemKey",
            "AmountLocal", "AmountEUR", "IsOpen"))

for nm_, e in [("order->delivery", edge_od), ("delivery->invoice", edge_di),
               ("invoice->receivable", edge_ir)]:
    print(f"{nm_:22s}{e.count():>8,} edges")

## 6. Gold contract gate, then write

Same gate as notebook 07: no `Decimal`, no unsafe column names. It runs **before** any write.

In [ ]:
NAME_RE = re.compile(r"^[A-Za-z][A-Za-z0-9_]*$")

def enforce_gold(df, name):
    problems = []
    for fld in df.schema.fields:
        if not NAME_RE.match(fld.name):
            problems.append(f"column name {fld.name!r} would trigger Delta column mapping")
        if isinstance(fld.dataType, T.DecimalType):
            problems.append(f"column {fld.name!r} is {fld.dataType.simpleString()} - "
                            "Fabric Graph returns null for Decimal")
    if problems:
        raise ValueError(f"{name} violates the Gold contract: " + "; ".join(problems))
    return df

TABLES = {
    "doc_delivery":              gld_delivery,
    "doc_invoice":               gld_invoice,
    "doc_receivable":            gld_receivable,
    "edge_order_to_delivery":    edge_od,
    "edge_delivery_to_invoice":  edge_di,
    "edge_invoice_to_receivable":edge_ir,
}

for name, df in TABLES.items():
    enforce_gold(df, name)
print("PASS  all six tables satisfy the Gold contract")

for name, df in TABLES.items():
    full = f"{GOLD}{name}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(full))
    print(f"{full:34s}{spark.table(full).count():>10,}")

## 7. Validation gate

The point of this notebook is that every edge resolves. An edge pointing at a missing node renders as a silently broken graph.

In [ ]:
failures = []

def check(label, ok, detail=""):
    print(f"{'PASS' if ok else 'FAIL'}  {label}" + (f"  - {detail}" if detail else ""))
    if not ok:
        failures.append(label)

o2c   = spark.table(f"{GOLD}fact_order_to_cash")
dlv_n = spark.table(f"{GOLD}doc_delivery")
inv_n = spark.table(f"{GOLD}doc_invoice")
rcv_n = spark.table(f"{GOLD}doc_receivable")
e_od  = spark.table(f"{GOLD}edge_order_to_delivery")
e_di  = spark.table(f"{GOLD}edge_delivery_to_invoice")
e_ir  = spark.table(f"{GOLD}edge_invoice_to_receivable")

# node keys unique
for label, df, key in [("delivery", dlv_n, "DeliveryId"),
                       ("invoice", inv_n, "BillingDocId"),
                       ("receivable", rcv_n, "ArItemKey")]:
    n, d = df.count(), df.select(key).distinct().count()
    check(f"{label} node key is unique", n == d, f"{n:,} rows / {d:,} keys")

# every edge endpoint resolves to a node
orphan_od_src = e_od.join(o2c, "OrderLineKey", "left_anti").count()
orphan_od_tgt = e_od.join(dlv_n, "DeliveryId", "left_anti").count()
orphan_di_src = e_di.join(dlv_n, "DeliveryId", "left_anti").count()
orphan_di_tgt = e_di.join(inv_n, "BillingDocId", "left_anti").count()
orphan_ir_src = e_ir.join(inv_n, "BillingDocId", "left_anti").count()
orphan_ir_tgt = e_ir.join(rcv_n, "ArItemKey", "left_anti").count()

check("order->delivery origin resolves to an order line", orphan_od_src == 0, f"{orphan_od_src:,} orphans")
check("order->delivery target resolves to a delivery",    orphan_od_tgt == 0, f"{orphan_od_tgt:,} orphans")
check("delivery->invoice origin resolves to a delivery",  orphan_di_src == 0, f"{orphan_di_src:,} orphans")
check("delivery->invoice target resolves to an invoice",  orphan_di_tgt == 0, f"{orphan_di_tgt:,} orphans")
check("invoice->receivable origin resolves to an invoice",orphan_ir_src == 0, f"{orphan_ir_src:,} orphans")
check("invoice->receivable target resolves to a receivable", orphan_ir_tgt == 0, f"{orphan_ir_tgt:,} orphans")

# the whole chain is walkable end to end
chain = (e_od.select("OrderLineKey", "DeliveryId")
    .join(e_di.select("DeliveryId", "BillingDocId"), "DeliveryId")
    .join(e_ir.select("BillingDocId", "ArItemKey"), "BillingDocId"))
walk = chain.count()
check("full OrderLine->Delivery->Invoice->Receivable chain is walkable", walk > 0, f"{walk:,} complete paths")
print(f"\ndistinct order lines reaching cash: {chain.select('OrderLineKey').distinct().count():,}")

# reconciliation against the existing document-flow table
flow_od = (spark.table(f"{GOLD}fact_document_flow")
           .filter((F.col("PrecedingCategory") == "C") & (F.col("SubsequentCategory") == "J")).count())
check("order->delivery edge count matches gld_fact_document_flow C->J",
      e_od.count() == flow_od, f"edges {e_od.count():,} vs flow {flow_od:,}")

assert not failures, f"{len(failures)} validation check(s) failed: {failures}"
print("\nALL CHECKS PASSED")

## 8. What to do next in Fabric

1. **Refresh the semantic model** if you add any of these tables to it (Direct Lake rebuilds
   relationship structures only on refresh - a new relationship queried before refresh returns
   *"does not hold any data because it needs to be recalculated"*).
2. **Ontology:** add `doc_delivery`, `doc_invoice`, `doc_receivable` as entity types and set their
   key properties to `DeliveryId`, `BillingDocId`, `ArItemKey`.
3. **Graph:** add three edges, each sourced from its edge table. See `Phase8/LINEAGE_STEPS.md`.

Reminder from Phase 4: if a cell fails with *"No default context found"*, the SQL analytics
endpoint was attached instead of the lakehouse. Remove it, re-add the row with the lighter-blue
lakehouse icon, accept *Stop current session*, and confirm the pin glyph.